In [1]:
library(ggplot2)
library(dplyr)
library(gridExtra)

df_meta <- read.csv("../Data_Metadata_Annotation.csv")
dup_cols <- c("Technology", "Tissue", "Dataset", "Sample")
df_meta <- df_meta %>% mutate(
  is_duplicated = duplicated(select(., all_of(dup_cols)))
)
df_duplicates <- df_meta %>% filter(is_duplicated)
df_meta_unique <- df_meta %>% filter(!is_duplicated) %>% select(-is_duplicated)

df_meta <- df_meta_unique


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine




In [3]:
df_meta

X,Technology,Tissue,Dataset,Sample,Number.of.Spots,Number.of.Genes,Genes.Detected.per.Spot,Sparsity,Avg.Expression.per.Spot,Avg.Non.Zero.Expression.per.Spot,Number.of.Clusters,Annotation_Level,Original_Annotation,Level
<int>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<chr>,<chr>
0,10x Visium,Brain,DLPFC,151507,4221,33538,1409.482,0.958,0.077,1.782,7,domain,Manual Annotation,Level 1
1,10x Visium,Brain,DLPFC,151508,4381,33538,1191.235,0.964,0.063,1.723,7,domain,Manual Annotation,Level 1
2,10x Visium,Brain,DLPFC,151509,4788,33538,1434.161,0.957,0.076,1.718,7,domain,Manual Annotation,Level 1
3,10x Visium,Brain,DLPFC,151510,4595,33538,1368.559,0.959,0.073,1.735,7,domain,Manual Annotation,Level 1
4,10x Visium,Brain,DLPFC,151669,3636,33538,1796.764,0.946,0.112,2.009,5,domain,Manual Annotation,Level 1
5,10x Visium,Brain,DLPFC,151670,3484,33538,1675.843,0.950,0.102,1.965,5,domain,Manual Annotation,Level 1
6,10x Visium,Brain,DLPFC,151671,4093,33538,1859.503,0.945,0.116,1.998,5,domain,Manual Annotation,Level 1
7,10x Visium,Brain,DLPFC,151672,3888,33538,1770.257,0.947,0.109,1.975,5,domain,Manual Annotation,Level 1
8,10x Visium,Brain,DLPFC,151673,3611,33538,2207.998,0.934,0.137,1.983,7,domain,Manual Annotation,Level 1


In [6]:
df_all

X,Technology,Tissue,Dataset,Sample,Number.of.Spots,Number.of.Genes,Genes.Detected.per.Spot,Sparsity,Avg.Expression.per.Spot,Avg.Non.Zero.Expression.per.Spot,Number.of.Clusters,Annotation_Level,Original_Annotation,Level
<int>,<fct>,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<chr>,<chr>
0,10x Visium,Brain,DLPFC,151507,4221,33538,1409.482,0.958,0.077,1.782,7,domain,Manual Annotation,Level 1
1,10x Visium,Brain,DLPFC,151508,4381,33538,1191.235,0.964,0.063,1.723,7,domain,Manual Annotation,Level 1
2,10x Visium,Brain,DLPFC,151509,4788,33538,1434.161,0.957,0.076,1.718,7,domain,Manual Annotation,Level 1
3,10x Visium,Brain,DLPFC,151510,4595,33538,1368.559,0.959,0.073,1.735,7,domain,Manual Annotation,Level 1
4,10x Visium,Brain,DLPFC,151669,3636,33538,1796.764,0.946,0.112,2.009,5,domain,Manual Annotation,Level 1
5,10x Visium,Brain,DLPFC,151670,3484,33538,1675.843,0.950,0.102,1.965,5,domain,Manual Annotation,Level 1
6,10x Visium,Brain,DLPFC,151671,4093,33538,1859.503,0.945,0.116,1.998,5,domain,Manual Annotation,Level 1
7,10x Visium,Brain,DLPFC,151672,3888,33538,1770.257,0.947,0.109,1.975,5,domain,Manual Annotation,Level 1
8,10x Visium,Brain,DLPFC,151673,3611,33538,2207.998,0.934,0.137,1.983,7,domain,Manual Annotation,Level 1


In [13]:
df_all <- df_meta
technology_order <- c('ST', '10x Visium', 'Slideseq', 'Stereoseq', 'VisiumHD', 
                      'seqFISH', 'STARmap', 'MERFISH', 'CosMx', 'Xenium')

technology_colors <- c(
  'ST' = '#FF7F0E',
  '10x Visium' = '#1F77B4', 
  'Slideseq' = '#2CA02C',
  'Stereoseq' = '#9467BD',
  'VisiumHD' = '#D62728',
  'seqFISH' = '#8C564B',
  'STARmap' = '#E377C2',
  'MERFISH' = '#7F7F7F',
  'CosMx' = '#BCBD22',
  'Xenium' = '#17BECF'
)

df_all$Technology <- factor(df_all$Technology, levels = technology_order)

summary_stats <- function(data, metric) {
  data %>%
    group_by(Technology) %>%
    summarise(
      mean_val = mean(.data[[metric]], na.rm = TRUE),
      sd_val = sd(.data[[metric]], na.rm = TRUE)
    )
}

plot_metric <- function(data, metric, xlabel, digits = 0, log_scale = FALSE) {
  stats <- summary_stats(data, metric)
  stats$Technology <- factor(stats$Technology, levels = rev(technology_order))
  if (digits == 0) {
    stats$label <- sprintf("%d", round(stats$mean_val))
  } else {
    stats$label <- sprintf(paste0("%.", digits, "f"), stats$mean_val)
  }
  
  p <- ggplot(stats, aes(x = mean_val, y = Technology, fill = Technology)) +
    geom_col(color = "black", width = 0.6) +
    geom_errorbarh(aes(xmin = mean_val - sd_val, xmax = mean_val + sd_val),
                   height = 0.3, size = 0.5, color = "black") +
    geom_text(aes(label = label), hjust = -0.1, size = 3) +
    scale_fill_manual(values = technology_colors) +
    labs(x = xlabel, y = NULL) +
    theme_minimal(base_size = 8) +
    theme(
      legend.position = "none",
      axis.text.y = element_text(size = 7),
      axis.text.x = element_text(size = 7),
      axis.title.x = element_text(size = 8),
      panel.grid.major.y = element_blank(),
      panel.grid.minor = element_blank()
    )
  
  if (log_scale) {
    p <- p + scale_x_log10() + annotation_logticks(sides = "b")
  }
  
  return(p)
}


p1 <- plot_metric(df_all, "Number.of.Spots", "Number of Spots", digits = 0, log_scale = TRUE)
p2 <- plot_metric(df_all, "Number.of.Genes", "Number of Genes", digits = 0, log_scale = TRUE)
p3 <- plot_metric(df_all, "Sparsity", "Sparsity", digits = 3, log_scale = FALSE)
pdf("plot_metrics.pdf", width = 18 / 2.54, height = 6 / 2.54)
grid.arrange(p1, p2, p3, ncol = 3)
dev.off()

png 
  2